In [105]:
# Instructions:
# Run `pip install requests`

from json import loads, dumps
import requests

status_codes = {
    200: 'OK',
    201: 'Created',
    400: 'Bad Request',
    401: 'Unauthorized',
    403: 'Forbidden',
    404: 'Not Found',
    409: 'Conflict',
    500: 'Internal Server Error'
}

def displayRequest(r: requests.models.Response):
    print(f"Status code: {r.status_code} ({status_codes.get(r.status_code, '')})")
    try:
        print("Reponse data:", loads(r.content))
    except Exception:
        print("Raw response data:", r.content)

In [106]:
# Some setup
some_needs = [
    {"name": "1000 Lines of Code", "description": "Created for some code project"},
    {"name": "100 Lines of Code", "description": "For a side-project"},
    {"name": "100000 Lines of Code", "description": "For Google.com"},
    {"name": "10 Lines of Code", "description": "For a bash script"},
    {"name": "5 Boxes of Cereal", "description": "For a food pantry"},
    {"name": "100$ Donation", "description": "Thanks for your support!"}
]

for need in some_needs:
    requests.post("http://127.0.0.1:8080/cupboard", dumps(need), headers={"Content-Type":"application/json"})

manager_account = {
    'username': 'admin',
    'password': 'adminpw'
}
requests.post("http://127.0.0.1:8080/user", json=manager_account)

<Response [201]>

In [108]:
# User story: Get a single need
req = requests.get("http://127.0.0.1:8080/cupboard/1")
displayRequest(req)

Status code: 200 (OK)
Reponse data: {'name': '1000 Lines of Code', 'id': 1, 'description': 'Created for some code project'}


In [109]:
# Creating a user
username = "team-123123123"
password = "hunter2"

my_account = {
    "username": username,
    "password": password
}

# Internal-only request
req = requests.post("http://127.0.0.1:8080/user", json=my_account)
displayRequest(req)
my_id = loads(req.text)['id']
my_id

Status code: 201 (Created)
Reponse data: {'id': 2, 'username': 'team-123123123', 'password': 'hunter2', 'basket': []}


2

In [110]:
# User story: Search for needs
# User story: View need details, for Manager or Helper

req = requests.get("http://127.0.0.1:8080/cupboard/?name=Code")
displayRequest(req)
for need in loads(req.content):
    print(need)

Status code: 200 (OK)
Reponse data: [{'name': '1000 Lines of Code', 'id': 1, 'description': 'Created for some code project'}, {'name': '100 Lines of Code', 'id': 2, 'description': 'For a side-project'}, {'name': '100000 Lines of Code', 'id': 3, 'description': 'For Google.com'}, {'name': '10 Lines of Code', 'id': 4, 'description': 'For a bash script'}]
{'name': '1000 Lines of Code', 'id': 1, 'description': 'Created for some code project'}
{'name': '100 Lines of Code', 'id': 2, 'description': 'For a side-project'}
{'name': '100000 Lines of Code', 'id': 3, 'description': 'For Google.com'}
{'name': '10 Lines of Code', 'id': 4, 'description': 'For a bash script'}


In [111]:
# User story: Add needs to basket

adding_data = {
    "userID": my_id,
    "needID": 201
}

req = requests.post("http://127.0.0.1:8080/user/basket/add", json=adding_data)
displayRequest(req)

# We don't have an API key yet!

Status code: 401 (Unauthorized)
Raw response data: b''


In [112]:
# User story: Log in Helpers

my_account = {
    "username": username,
    "password": password
}

req = requests.post("http://127.0.0.1:8080/accounts/login", json=my_account)
displayRequest(req)
key = req.text

Status code: 200 (OK)
Raw response data: b'0dd5cee9eec495df7a62478065732670'


In [113]:
# User story: Add needs to basket

adding_data = {
    "userID": my_id,
    "needID": 2
}

req = requests.post("http://127.0.0.1:8080/user/basket/add", json=adding_data, headers={"key": key})
displayRequest(req)

# Let's add a few more
req = requests.post("http://127.0.0.1:8080/user/basket/add", json={"userID": my_id, "needID": 1}, headers={"key": key})
req = requests.post("http://127.0.0.1:8080/user/basket/add", json={"userID": my_id, "needID": 4}, headers={"key": key})
req = requests.post("http://127.0.0.1:8080/user/basket/add", json={"userID": my_id, "needID": 3}, headers={"key": key})

Status code: 200 (OK)
Reponse data: {'id': 2, 'username': 'team-123123123', 'password': 'hunter2', 'basket': [2]}


In [114]:
# User story: Remove needs from basket

req = requests.post("http://127.0.0.1:8080/user/basket/remove", json={"userID": my_id, "needID": 3}, headers={"key": key})
displayRequest(req)

Status code: 200 (OK)
Reponse data: {'id': 2, 'username': 'team-123123123', 'password': 'hunter2', 'basket': [2, 1, 4]}


In [115]:
# User story: View basket

req = requests.get(f"http://127.0.0.1:8080/user/basket/{my_id}", headers={"key": key})
displayRequest(req)

Status code: 200 (OK)
Reponse data: [2, 1, 4]


In [116]:
# User story: Checkout needs

req = requests.post(f"http://127.0.0.1:8080/user/basket/checkout/{my_id}", headers={"key": key})
displayRequest(req)

Status code: 200 (OK)
Reponse data: {'id': 2, 'username': 'team-123123123', 'password': 'hunter2', 'basket': []}


In [117]:
# User story: Get all needs

req = requests.get("http://127.0.0.1:8080/cupboard")
displayRequest(req)
print("Needs found:")
for need in loads(req.content):
    print(need)

Status code: 200 (OK)
Reponse data: [{'name': '100000 Lines of Code', 'id': 3, 'description': 'For Google.com'}, {'name': '5 Boxes of Cereal', 'id': 5, 'description': 'For a food pantry'}, {'name': '100$ Donation', 'id': 6, 'description': 'Thanks for your support!'}]
Needs found:
{'name': '100000 Lines of Code', 'id': 3, 'description': 'For Google.com'}
{'name': '5 Boxes of Cereal', 'id': 5, 'description': 'For a food pantry'}
{'name': '100$ Donation', 'id': 6, 'description': 'Thanks for your support!'}


In [118]:
# User story: Log out user

req = requests.post("http://127.0.0.1:8080/accounts/logout", json=my_account)
displayRequest(req)

Status code: 200 (OK)
Raw response data: b''


In [128]:
# User story: Log in Manager

req = requests.post("http://127.0.0.1:8080/accounts/login", json=manager_account)
displayRequest(req)
manager_key = req.text

Status code: 200 (OK)
Raw response data: b'3fbb30d4b37d3e0f00a66680860565a9'


In [120]:
# User story: Create new need
new_need = {"name": "Canned Soup", "description": "Chicken Noodle"}

req = requests.post("http://127.0.0.1:8080/manager/add", json=new_need, headers={"key": manager_key})
displayRequest(req)

Status code: 200 (OK)
Reponse data: {'name': 'Canned Soup', 'id': 7, 'description': 'Chicken Noodle'}


In [121]:
req = requests.get("http://127.0.0.1:8080/cupboard")
displayRequest(req)
print("Needs found:")
for need in loads(req.content):
    print(need)

Status code: 200 (OK)
Reponse data: [{'name': '100000 Lines of Code', 'id': 3, 'description': 'For Google.com'}, {'name': '5 Boxes of Cereal', 'id': 5, 'description': 'For a food pantry'}, {'name': '100$ Donation', 'id': 6, 'description': 'Thanks for your support!'}, {'name': 'Canned Soup', 'id': 7, 'description': 'Chicken Noodle'}]
Needs found:
{'name': '100000 Lines of Code', 'id': 3, 'description': 'For Google.com'}
{'name': '5 Boxes of Cereal', 'id': 5, 'description': 'For a food pantry'}
{'name': '100$ Donation', 'id': 6, 'description': 'Thanks for your support!'}
{'name': 'Canned Soup', 'id': 7, 'description': 'Chicken Noodle'}


In [ ]:
# User story: Remove need, Remove need from Cupboard

req = requests.post("http://127.0.0.1:8080/manager/delete/3", headers={"key": manager_key})
displayRequest(req)

Status code: 200 (OK)
Raw response data: b''


In [129]:
# User story: Edit need, Update need in Cupboard
new_need_data = {
    "id": 7,
    "description": "Chicken Noodle, 5 cans"
}

req = requests.post("http://127.0.0.1:8080/manager/edit", json=new_need_data, headers={"key": manager_key})
displayRequest(req)

Status code: 200 (OK)
Reponse data: {'name': None, 'id': 7, 'description': 'Chicken Noodle, 5 cans'}
